In [19]:
import yfinance as yf
import pandas as pd
import numpy as np
import yahooquery as yq
import sys
sys.path.append("../")
from src import calculations,data_cleaning,data_import,name_to_Ticker,plots,data_cleaning


Close
Daily_Return
Cumulative_Return
SMA_20
SMA_50
SMA_200
EMA_20
EMA_50
EMA_200
RSI_14
ATR_14
MACD
MACD_Signal
MACD_Histogram
OBV

In [21]:
df=yf.Ticker("MSFT").history(period="1y")
df=data_cleaning.data_clean(df)
price_metrics=calculations.calculate_price_metrics(df)
moving_averages=calculations.calculate_moving_averages(df)
rsi=calculations.calc_rsi(df)
atr=calculations.calculate_atr(df)
macd=calculations.calculate_macd(df)
volume=calculations.volume_metrics(df)

In [24]:
calculated_df=pd.DataFrame()
calculated_df=pd.concat([df["Close"],price_metrics["daily_return"],price_metrics["cumulative_return"],moving_averages,rsi,atr,macd,volume["OBV"]],axis=1)
calculated_df

,Close,daily_return,cumulative_return,SMA_20,SMA_50,SMA_200,EMA_20,EMA_50,EMA_200,RSI,atr,MACD,MACD_SIGNAL,MACD_HISTOGRAM,OBV
Date,,,,,,,,,,,,,,,
2025-08-20 00:00:00-04:00,501.712555,NaN,0.000000,NaN,NaN,NaN,501.712555,501.712555,501.712555,NaN,NaN,0.000000,0.000000,0.000000,27723000.0
2025-08-21 00:00:00-04:00,501.066620,-0.001287,-0.001287,NaN,NaN,NaN,501.651037,501.687224,501.706128,NaN,NaN,-0.051528,-0.010306,-0.041222,9279700.0
2025-08-22 00:00:00-04:00,504.037842,0.005930,0.004635,NaN,NaN,NaN,501.878352,501.779405,501.729329,NaN,NaN,0.145710,0.020897,0.124812,33603900.0
2025-08-25 00:00:00-04:00,501.086517,-0.005855,-0.001248,NaN,NaN,NaN,501.802939,501.752233,501.722933,NaN,NaN,0.063146,0.029347,0.033799,11965300.0
2025-08-26 00:00:00-04:00,498.880463,-0.004403,-0.005645,NaN,NaN,NaN,501.524608,501.639615,501.694649,NaN,NaN,-0.178241,-0.012170,-0.166071,-18870400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-14 00:00:00-04:00,495.399994,-0.002979,-0.012582,450.240999,412.775598,431.316493,462.018677,431.221535,432.835557,70.923506,13.802692,28.591420,25.073823,3.517597,148628100.0
2026-08-17 00:00:00-04:00,480.350006,-0.030379,-0.042579,454.143999,413.821598,431.024951,463.764517,433.148142,433.308338,62.197372,14.030356,26.460878,25.351234,1.109644,118992300.0
2026-08-18 00:00:00-04:00,481.630005,0.002665,-0.040028,458.337999,415.120798,430.742392,465.465992,435.049391,433.789151,62.618627,13.536759,24.592209,25.199429,-0.607220,143079900.0


In [25]:
calculated_df.columns

Index(['Close', 'daily_return', 'cumulative_return', 'SMA_20', 'SMA_50',
       'SMA_200', 'EMA_20', 'EMA_50', 'EMA_200', 'RSI', 'atr', 'MACD',
       'MACD_SIGNAL', 'MACD_HISTOGRAM', 'OBV'],
      dtype='str')

In [ ]:
latest=calculated_df.iloc[-1]

Close                4.806697e+02
daily_return        -7.516445e-03
cumulative_return   -4.194204e-02
SMA_20               4.679910e+02
SMA_50               4.181174e+02
SMA_200              4.303823e+02
EMA_20               4.685377e+02
EMA_50               4.386945e+02
EMA_200              4.347533e+02
RSI                  6.133572e+01
atr                  1.267486e+01
MACD                 2.130936e+01
MACD_SIGNAL          2.407938e+01
MACD_HISTOGRAM      -2.770019e+00
OBV                  1.554906e+08
Name: 2026-08-20 00:00:00-04:00, dtype: float64

In [28]:
latest["RSI"]<30

np.False_

In [29]:
latest["Close"] > latest["SMA_200"]

np.True_

In [30]:
latest["MACD"] > latest["MACD_SIGNAL"]

np.False_

In [31]:
latest["OBV"] > 0

np.True_

In [32]:
(
    (latest["RSI"] < 30)
    & (latest["Close"] > latest["SMA_200"])
    & (latest["MACD"] > latest["MACD_SIGNAL"])
)

np.False_

In [39]:
def apply_condition(df, metric, operator, value):

    latest = df.iloc[-1]
    if isinstance(value, str):
        value = latest[value]

    if operator == ">":
        return latest[metric] > value

    elif operator == "<":
        return latest[metric] < value

    elif operator == ">=":
        return latest[metric] >= value

    elif operator == "<=":
        return latest[metric] <= value

    elif operator == "==":
        return latest[metric] == value

    elif operator == "!=":
        return latest[metric] != value

    else:
        raise ValueError("Invalid operator")

def screen_stock(df, conditions):

    for metric, operator, value in conditions:

        if not apply_condition(df, metric, operator, value):
            return False

    return True
condition=[("RSI","<",30),("RSI",">","SMA20")]
screen_stock(calculated_df,condition)

False

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA", "AMZN"]
conditions = [
    ("RSI", "<", 30),
    ("Close", ">", "SMA_200"),
    ("MACD", ">", "MACD_SIGNAL")
]